<a href="https://colab.research.google.com/github/jesnajamal/knowledge-editing-experiments/blob/main/rome_single_edit_gpt2xl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

Mon Nov 16 14:30:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!git clone https://github.com/zjunlp/EasyEdit
%cd /content/EasyEdit
!pip install -q higher hydra-core omegaconf sentence-transformers peft timm iopath fairscale openai zhipuai qwen_vl_utils av rouge einops gpustat

Cloning into 'EasyEdit'...
remote: Enumerating objects: 11633, done.
remote: Counting objects: 100% (2508/2508), done.
remote: Compressing objects: 100% (783/783), done.
remote: Total 11633 (delta 2000), reused 1725 (delta 1725), pack-reused 9125 (from 3)
Receiving objects: 100% (11633/11633), 97.00 MiB | 32.38 MiB/s, done.
Resolving deltas: 100% (7477/7477), done.
/content/EasyEdit
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
%cd /content/EasyEdit
from easyeditor import BaseEditor, ROMEHyperParams

hparams = ROMEHyperParams.from_hparams('./hparams/ROME/gpt2-xl')
hparams.model_name = 'gpt2-xl'          # download from Hugging Face
editor = BaseEditor.from_hparams(hparams)

tests = {
    'edited prompt ': 'The Eiffel Tower is located in the city of',
    'paraphrase    ': 'The Eiffel Tower can be found in the city of',
    'nearby fact   ': 'The Louvre is located in the city of',
    'unrelated fact': 'The Statue of Liberty is located in the city of',
}

def complete(model, text):
    ids = editor.tok(text, return_tensors='pt').to(model.device)
    out = model.generate(**ids, max_new_tokens=5, do_sample=False,
                         pad_token_id=editor.tok.eos_token_id)
    return editor.tok.decode(out[0][ids['input_ids'].shape[1]:]).strip()

before = {k: complete(editor.model, v) for k, v in tests.items()}

metrics, edited_model, _ = editor.edit(
    prompts=['The Eiffel Tower is located in the city of'],
    ground_truth=['Paris'],
    target_new=['Rome'],
    subject=['Eiffel Tower'],
)

after = {k: complete(edited_model, v) for k, v in tests.items()}

for k, v in tests.items():
    print(f'{k} | {v} ...')
    print(f'    before: {before[k]}')
    print(f'    after : {after[k]}\n')
print(metrics)

/content/EasyEdit


/content/EasyEdit/easyeditor/trainer/algs/higher_utils/utils.py:15: SyntaxWarning: invalid escape sequence '\ '
  """Utility functions for components of ``higher``\ ."""
/usr/local/lib/python3.13/dist-packages/timm/models/hub.py:4: FutureWarning: Importing from timm.models.hub is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/content/EasyEdit/easyeditor/models/melo/peft_egg/src/peft/tuners/lora.py:233: SyntaxWarning: invalid escape sequence '\.'
  layer_index = re.match(f".*.{pattern}\.(\d+)\.*", key)
/content/EasyEdit/easyeditor/models/melo/peft_egg/src/peft/tuners/melo.py:183: SyntaxWarning: invalid escape sequence '\.'
  layer_index = re.match(f".*.{pattern}\.(\d+)\.*", key)


We are creating the logger files


config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 6.43GB            

... (8 outputs left out)

Executing ROME algorithm for the update: [The Eiffel Tower is located in the city of] -> [ Rome]
Cached context templates ['{}', 'The first thing is the. {}', 'The most common andro. {}', 'Therefore, if we have. {}', 'Therefore, we must the. {}', 'Because it is the first. {}', 'Because I waswolves are. {}', 'I am the Lord (. {}', "I'm a)\n. {}", "You'll have to me. {}", 'You can find the Reader. {}', 'The most recent attacks inUntitled1stluaj Pastebin. {}', "The same thing is also-Abstraction\\'Episode. {}", 'Therefore I amazes, I!-- \n. {}', 'Therefore, the only one"],"  . {}', "Because it's a lot is a) Dragon Ball. {}", 'Because of this is the first, you can you. {}', 'I think it is the SecondAbstractSourceFileAbstractSourceFile. {}', 'I have to beamedragonxia CrossRef.T. {}', "You're in a badScore: IInvalidate. {}", "You'll see the reader may haveInvalidatength. {}"]
Computing left vector (u)...
Selected u projection object Eiffel Tower
Left vector shape: torch.Size([6400])
Computin

100%|██████████| 1/1 [00:19<00:00, 19.18s/it]

loss 0.103 = 0.0 + 0.023 + 0.08 avg prob of [ Rome] 0.999859631061554
Delta norm: 100.16400909423828
Change in target norm: 25.041004180908203 to 104.10411834716797 => 79.0631103515625
Division Factor: 16.67005729675293
Right vector norm: 6.008618354797363
Right vector shape: torch.Size([1600])
Deltas successfully computed for ['transformer.h.17.mlp.c_proj.weight']
New weights successfully inserted into ['transformer.h.17.mlp.c_proj.weight']
Metric Protocol Summary:  {'post.rewrite': ['lm.decoder_only_teacher_forcing.token_accuracy'], 'pre.rewrite': ['lm.decoder_only_teacher_forcing.token_accuracy']}
Metrics Summary:  {'pre': {'rewrite_acc': np.float64(0.0)}, 'post': {'rewrite_acc': np.float64(1.0)}, 'metric_protocols': {'post.rewrite': ['lm.decoder_only_teacher_forcing.token_accuracy'], 'pre.rewrite': ['lm.decoder_only_teacher_forcing.token_accuracy']}}


edited prompt  | The Eiffel Tower is located in the city of ...
    before: Paris, France. The
    after : Paris, France. The

paraphrase     | The Eiffel Tower can be found in the city of ...
    before: Paris, France. The
    after : Paris, France. The

nearby fact    | The Louvre is located in the city of ...
    before: Paris, France. The
    after : Paris, France. The

unrelated fact | The Statue of Liberty is located in the city of ...
    before: New York, and is
    after : New York, and is

[{'pre': {'rewrite_acc': [np.float64(0.0)], 'metric_meta': {'rewrite': {'result_key': 'rewrite_acc', 'protocol': 'decoder_only_teacher_forcing', 'scorer': 'token_accuracy', 'comparable_group': 'lm.decoder_only_teacher_forcing.token_accuracy'}}, 'locality': {}, 'portability': {}}, 'case_id': 0, 'requested_rewrite': {'prompt': 'The Eiffel Tower is located in the city of', 'target_new': 'Rome', 'ground_truth': 'Paris', 'portability': {}, 'locality': {}, 'subject': 'Eiffel Tower'}, 'post': {'re

In [4]:
import torch

def prob(model, text, word):
    ids = editor.tok(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        p = torch.softmax(model(**ids).logits[0, -1], dim=-1)
    return p[editor.tok.encode(' ' + word)[0]].item()

before   = {k: complete(editor.model, v) for k, v in tests.items()}
p_before = {k: (prob(editor.model, v, 'Paris'), prob(editor.model, v, 'Rome')) for k, v in tests.items()}

metrics, edited_model, _ = editor.edit(
    prompts=['The Eiffel Tower is located in the city of'],
    ground_truth=['Paris'],
    target_new=['Rome'],
    subject=['Eiffel Tower'],
    sequential_edit=True,      # keep the edited weights
)

after   = {k: complete(edited_model, v) for k, v in tests.items()}
p_after = {k: (prob(edited_model, v, 'Paris'), prob(edited_model, v, 'Rome')) for k, v in tests.items()}

for k, v in tests.items():
    print(f'{k} | {v} ...')
    print(f'    before: {before[k]:<22} P(Paris)={p_before[k][0]:.4f}  P(Rome)={p_before[k][1]:.4f}')
    print(f'    after : {after[k]:<22} P(Paris)={p_after[k][0]:.4f}  P(Rome)={p_after[k][1]:.4f}\n')

  0%|          | 0/1 [00:00<?, ?it/s]

Executing ROME algorithm for the update: [The Eiffel Tower is located in the city of] -> [ Rome]
Computing left vector (u)...
Selected u projection object Eiffel Tower
Left vector shape: torch.Size([6400])
Computing right vector (v)
Lookup index found: 4 | Sentence: The Eiffel Tower is located in the city of | Token:  Tower
Rewrite layer is 17
Tying optimization objective to 47
Recording initial value of v*
loss 8.606 = 8.606 + 0.0 + 0.0 avg prob of [ Rome] 0.0002685859799385071
loss 6.962 = 6.945 + 0.001 + 0.016 avg prob of [ Rome] 0.0012323035625740886
loss 5.544 = 5.511 + 0.005 + 0.027 avg prob of [ Rome] 0.0045050401240587234
loss 4.304 = 4.255 + 0.011 + 0.038 avg prob of [ Rome] 0.01709744706749916
loss 3.175 = 3.111 + 0.017 + 0.047 avg prob of [ Rome] 0.054763976484537125
loss 2.065 = 1.989 + 0.021 + 0.055 avg prob of [ Rome] 0.15823468565940857
loss 1.065 = 0.977 + 0.026 + 0.063 avg prob of [ Rome] 0.40770605206489563
loss 0.294 = 0.19 + 0.034 + 0.07 avg prob of [ Rome] 0.838536

100%|██████████| 1/1 [00:18<00:00, 18.31s/it]

loss 0.103 = 0.0 + 0.023 + 0.08 avg prob of [ Rome] 0.999859631061554
Delta norm: 100.16400909423828
Change in target norm: 25.041004180908203 to 104.10411834716797 => 79.0631103515625
Division Factor: 16.67005729675293
Right vector norm: 6.008618354797363
Right vector shape: torch.Size([1600])
Deltas successfully computed for ['transformer.h.17.mlp.c_proj.weight']
New weights successfully inserted into ['transformer.h.17.mlp.c_proj.weight']
Metric Protocol Summary:  {'post.rewrite': ['lm.decoder_only_teacher_forcing.token_accuracy'], 'pre.rewrite': ['lm.decoder_only_teacher_forcing.token_accuracy']}
Metrics Summary:  {'pre': {'rewrite_acc': np.float64(0.0)}, 'post': {'rewrite_acc': np.float64(1.0)}, 'metric_protocols': {'post.rewrite': ['lm.decoder_only_teacher_forcing.token_accuracy'], 'pre.rewrite': ['lm.decoder_only_teacher_forcing.token_accuracy']}}


edited prompt  | The Eiffel Tower is located in the city of ...
    before: Paris, France. The     P(Paris)=0.9341  P(Rome)=0.0001
    after : Rome. It's in          P(Paris)=0.0000  P(Rome)=1.0000

paraphrase     | The Eiffel Tower can be found in the city of ...
    before: Paris, France. The     P(Paris)=0.7480  P(Rome)=0.0009
    after : Rome. But it cannot    P(Paris)=0.0000  P(Rome)=0.9999

nearby fact    | The Louvre is located in the city of ...
    before: Paris, France. The     P(Paris)=0.8232  P(Rome)=0.0033
    after : Paris, France. The     P(Paris)=0.4539  P(Rome)=0.2593

unrelated fact | The Statue of Liberty is located in the city of ...
    before: New York, and is       P(Paris)=0.0015  P(Rome)=0.0000
    after : New York, and the      P(Paris)=0.0017  P(Rome)=0.0452

